In [7]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="smartsupply-ai")

query = """
SELECT *
FROM `smartsupply-ai.smartsupply.shipments`
"""

df = client.query(query).to_dataframe()

In [8]:
df = df.copy()

df['delay_days'] = (
    pd.to_datetime(df['Delivered to Client Date']) -
    pd.to_datetime(df['Scheduled Delivery Date'])
).dt.days

df['risk_flag'] = df['delay_days'].apply(lambda x: 1 if x > 3 else 0)

df[['delay_days', 'risk_flag']].head()

,delay_days,risk_flag
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Features (input)
X = df[['Unit Price', 'Line Item Quantity']]

# Target (output)
y = df['risk_flag']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Model trained successfully 🚀")

Model trained successfully 🚀


In [10]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8653753026634382
